# Salary Data Analysis Project

In [ ]:
import pandas as pd
import numpy as np
from numpy import nan as NA

# Load dataset
salary_df = pd.read_csv('salary.csv')
print(salary_df.shape)
print(salary_df.size)

## Step 1: Rename Multiple Columns

In [ ]:
salary_df.rename(columns={
    "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent what you would earn if you worked the job 40 hours a week, 52 weeks a year.)" : "Annual_salary",
    "How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.": "Additional_salary",
    "Please indicate the currency": "Currency",
    "How old are you?": "Age",
    "What is your gender?": "Sex"
}, inplace = True)

salary_df.head()

## Step 2: Drop Duplicate Rows

In [ ]:
salary_df.drop_duplicates(['Annual_salary', 'Additional_salary'], keep='last')

## Step 3: Fill null value with 0 in specific column

In [ ]:
salary_df_With_Add_Sal_0 = salary_df.fillna({'Additional_salary' : 0})
salary_df_With_Add_Sal_0.head()

## Step 4: Drop null value from specific column

In [ ]:
Drop_null_from_ann_salary = salary_df_With_Add_Sal_0.dropna(subset=['Annual_salary'])

# check is there any null values in specific column:
print(salary_df_With_Add_Sal_0['Additional_salary'].isnull().any())
print(Drop_null_from_ann_salary['Annual_salary'].isnull().any())

## CHECK DATA TYPE
print(Drop_null_from_ann_salary['Annual_salary'].dtype) #object
print(Drop_null_from_ann_salary['Additional_salary'].dtype) #float

## Step 5: String Manipulation

In [ ]:
Drop_null_from_ann_salary['Annual_salary'] = [float(str(i).replace(",", "")) for i in Drop_null_from_ann_salary['Annual_salary']]
print(Drop_null_from_ann_salary['Annual_salary'].dtype) #converted into float

## Step 6: Create a new column with Apply function for Total Salary

In [ ]:
def calculate_total_salary(x):
    return (x['Annual_salary']) + (x['Additional_salary'])

Drop_null_from_ann_salary['Total_Salary'] = Drop_null_from_ann_salary.apply(calculate_total_salary, axis=1)
Drop_null_from_ann_salary.head()

## Step 7: Only see the specific values from a specific column

In [ ]:
New_salary_df = Drop_null_from_ann_salary[Drop_null_from_ann_salary['Currency'].isin(['USD', 'CAD', 'EUR'])]
New_salary_df['Currency'].value_counts()

## Step 8: All Currencies converted into BDT in a new column

In [ ]:
exchange_rates = {'USD': 119.43, 'CAD': 83.17, 'Euro': 124.86, 'EUR': 124.86} #Define exchange rates

# Creating a clean copy to avoid SettingWithCopyWarning
New_salary_df = New_salary_df.copy()
New_salary_df['Exchange_Rate'] = New_salary_df['Currency'].map(exchange_rates) #Create a column for the exchange rates
New_salary_df['Equivalent_BDT'] = New_salary_df['Total_Salary'] * New_salary_df['Exchange_Rate'] #Calculate the converted amount in BDT

## CREATE A NEW DF
New_Salary_df = New_salary_df.loc[:, ['Age', 'Annual_salary', 'Additional_salary', 'Currency', 'Total_Salary', 'Equivalent_BDT']]
print(New_Salary_df['Equivalent_BDT'].isnull().any())

##fill null values with 0 in Equivalent_BDT column to do mean:
New_Salary_df = New_Salary_df.fillna({'Equivalent_BDT' : 0})

## Step 9: Discretization & Binning

In [ ]:
# replace the value:
New_Salary_df['Age'] = New_Salary_df['Age'].replace('65 or over', '65-100')
New_Salary_df['Age'] = New_Salary_df['Age'].replace('under 18', '0-17')

# converted age into int form/numeric form:
New_Salary_df['Age_num'] = New_Salary_df['Age'].astype(str).str.split('-').str[0].astype(int)

bins = [-1, 17, 34, 54, 64, float('inf')]
labels = ['0-17', '18-34', '35-54', '55-64', '65+']
New_Salary_df['Age_Bin'] = pd.cut(New_Salary_df['Age_num'], bins=bins, labels=labels, right=True)

print(New_Salary_df['Equivalent_BDT'].isnull().any())
print(New_Salary_df['Equivalent_BDT'].dtype)

## Step 10: Create an age category

In [ ]:
def categorize_age(x):
    age_Category = {
        '0-17': 1,
        '18-34': 2,
        '35-54': 3,
        '55-64': 4,
        '65+': 5
    }
    return age_Category.get(x)

New_Salary_df['Age_Category'] = New_Salary_df['Age_Bin'].astype(str).apply(categorize_age)

avg = New_Salary_df.groupby('Age_Category')['Equivalent_BDT'].mean()
highest_avg_salary = avg.sort_values(ascending=False)
print(highest_avg_salary)

Average_salary = New_Salary_df.groupby(['Age_Category']).agg(
    highest_avg = pd.NamedAgg(column="Equivalent_BDT", aggfunc=np.mean)
)
print(Average_salary)

# Summary of Totals and Means
for cat in range(1, 6):
    cat_sum = New_Salary_df[New_Salary_df['Age_Category'] == cat]['Equivalent_BDT'].sum()
    cat_mean = New_Salary_df[New_Salary_df['Age_Category'] == cat]['Equivalent_BDT'].mean()
    print(f"Category {cat} Sum:", cat_sum)
    print(f"Category {cat} Mean:", cat_mean)